In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # add/update this — lets it use full available width instead of guessing terminal width
pd.set_option('display.max_rows', None)

In [2]:
data = pd.read_csv("C:\\Users\\sulem\\OneDrive\\Desktop\\Codes\\ML\\Datasets\\diabetes.csv")
data.head()

Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  DiabetesPedigreeFunction  Age  Outcome
0            6      148             72             35        0  33.6                     0.627   50        1
1            1       85             66             29        0  26.6                     0.351   31        0
2            8      183             64              0        0  23.3                     0.672   32        1
3            1       89             66             23       94  28.1                     0.167   21        0
4            0      137             40             35      168  43.1                     2.288   33        1

In [3]:
data.corr()["Outcome"]

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x = data.drop("Outcome", axis=1)
y = data["Outcome"]

x = scaler.fit_transform(x)

trainx, testx, trainy, testy = train_test_split(x, y, random_state=33, test_size=0.3, stratify=y)

In [5]:
import keras_tuner as kt
from tensorflow import keras # type: ignore


# Define the model-building function

1. KerasTuner calls this function repeatedly, passing it a fresh
2. "hp" (HyperParameters) objects each trial. Every hp.Int, hp.Choice, hp.Float call defines one axis of the search space.

In [ ]:
def build_model(hp):
    model = keras.Sequential()
 
    # Input layer — shape must match number of features
    model.add(keras.layers.Input(shape=(trainx.shape[1],)))
 
    # Tune the NUMBER of hidden layers (between 1 and 3)
    for i in range(hp.Int("num of layers", min_value=1, max_value=4, step=1)):
 
        # Tune the number of nuerons in each hidden layer
        nuerons = hp.Int(
            "nuerons_{}".format(i), # unique per-layer name, e.g. "units_0", "units_1" so each layer nuerons get tuned differetly.
            min_value=8,
            max_value=128,
            step=8
        )
 
        # Tune the activation function used in each hidden layer
        activation = hp.Choice(
            "activation_{}".format(i), # unique per-layer name so each layer's activation is tuned separately
            values=["relu", "tanh"]
        )
 
        model.add(keras.layers.Dense(units=nuerons, activation=activation))
 
        # Tune whether/how much dropout to apply after each layer
        dropout_rate = hp.Float(
            "dropout_{}".format(i), # unique per-layer name so each layer's dropout rate is tuned separately
            min_value=0.0,
            max_value=0.5,
            step=0.1
        )

        if dropout_rate > 0:
            model.add(keras.layers.Dropout(dropout_rate))
 
    # Output layer — binary classification, so 1 unit + sigmoid
    model.add(keras.layers.Dense(1, activation="sigmoid"))
 
    # Tune the learning rate on a log scale
    learning_rate = hp.Float(
        "learning_rate",
        min_value=1e-4,
        max_value=1e-2,
        sampling="log"
    )
 
    # Tune which optimizer to use
    optimizer_choice = hp.Choice("optimizer", values=["adam", "rmsprop"])
    if optimizer_choice == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
 
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
 
    return model
 
# 3. Set up the tuner (RandomSearch)

tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",   # metric the tuner tries to maximize
    max_trials=10,               # how many different hyperparameter combos to try
    executions_per_trial=1,      # how many times to train each combo (avg out noise)
    # directory="kt_dir",          where trial logs/checkpoints are stored
    # project_name="random_search_demo",
    overwrite=True               # start a fresh search each run
)
 
# See a summary of the search space before running
tuner.search_space_summary()

# 4. Run the search
# Early stopping so bad trials don't waste time training to completion

stop_early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3)
 
tuner.search(
    trainx, trainy,
    validation_data=(testx, testy),
    epochs=100,
    batch_size=32,
    callbacks=[stop_early],
    verbose=1
)

import shutil

# after tuner.search() and after you've retrieved best_hp / best_model
shutil.rmtree("untitled_project", ignore_errors=True)

Trial 10 Complete [00h 00m 07s]
val_accuracy: 0.7965368032455444

Best val_accuracy So Far: 0.7965368032455444
Total elapsed time: 00h 01m 09s


In [ ]:
# Retrieve the best hyperparameters and best model

best_hyperparameter = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest hyperparameters found:")

for param, value in best_hyperparameter.values.items():
    print("  ", param, ":", value)


Best hyperparameters found:
   num of layers : 4
   nuerons_0 : 56
   activation_0 : relu
   dropout_0 : 0.1
   learning_rate : 0.0003455443690378424
   optimizer : adam
   nuerons_1 : 24
   activation_1 : tanh
   dropout_1 : 0.4
   nuerons_2 : 112
   activation_2 : tanh
   dropout_2 : 0.1
   nuerons_3 : 8
   activation_3 : relu
   dropout_3 : 0.0


In [13]:
best_model = build_model(best_hyperparameter)

history = best_model.fit(
    trainx, trainy,
    validation_data=(testx, testy),
    epochs=50,
    batch_size=32,
    callbacks=[stop_early],
    verbose=1
)

# Evaluate on the test set
test_loss, test_acc = best_model.evaluate(testx, testy, verbose=1)
print("\nTest accuracy:", round(test_acc, 4))
print("Test loss:     ", round(test_loss, 4))

Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.3966 - loss: 0.7380 - val_accuracy: 0.5368 - val_loss: 0.6879
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5698 - loss: 0.6742 - val_accuracy: 0.6580 - val_loss: 0.6452
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6518 - loss: 0.6424 - val_accuracy: 0.6797 - val_loss: 0.6159
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6797 - loss: 0.6165 - val_accuracy: 0.6840 - val_loss: 0.5940
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6853 - loss: 0.5802 - val_accuracy: 0.6883 - val_loss: 0.5791
Epoch 6/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7020 - loss: 0.5684 - val_accuracy: 0.7013 - val_loss: 0.5681
Epoch 7/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7207 - loss: 0.5562 - val_accuracy: 0.7143 - val_loss: 0.5571
Epoch 8/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7374 - loss: 0.5488 - val_accuracy: 0.7489 - v

In [14]:
# Hyperband is generally faster than RandomSearch because it allocates more epochs to promising trials and kills bad ones early.

hyperband_tuner = kt.Hyperband(
    hypermodel=build_model,
    objective="val_accuracy",
    max_epochs=30,
    factor=3,                  # how aggressively trials are pruned each round
    # directory="kt_dir",
    # project_name="hyperband_demo",
    overwrite=True
)

hyperband_tuner.search(
    trainx, trainy,
    validation_data=(testx, testy),
    callbacks=[stop_early],
    verbose=1
)

import shutil

# after tuner.search() and after you've retrieved best_hp / best_model
shutil.rmtree("untitled_project", ignore_errors=True)

Trial 84 Complete [00h 00m 06s]
val_accuracy: 0.7748917937278748

Best val_accuracy So Far: 0.8008658289909363
Total elapsed time: 00h 07m 36s


In [16]:
# Retrieve the best hyperparameters and best model

best_hyperparameter_hyperband = hyperband_tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest hyperparameters found:")

for param, value in best_hyperparameter_hyperband.values.items():
    print("  ", param, ":", value)

print("\n")

best_model_hyperband = build_model(best_hyperparameter_hyperband)

hyperband_history = best_model_hyperband.fit(
    trainx, trainy,
    validation_data=(testx, testy),
    epochs=50,
    batch_size=32,
    callbacks=[stop_early],
    verbose=1
)

# Evaluate on the test set

hyperband_test_loss, hyperband_test_acc = best_model_hyperband.evaluate(testx, testy, verbose=1)

print("\nTest accuracy:", round(test_acc, 4))
print("Test loss:     ", round(test_loss, 4))



Best hyperparameters found:
   num of layers : 4
   nuerons_0 : 64
   activation_0 : tanh
   dropout_0 : 0.2
   learning_rate : 0.00023681845901196664
   optimizer : rmsprop
   nuerons_1 : 64
   activation_1 : tanh
   dropout_1 : 0.0
   nuerons_2 : 104
   activation_2 : tanh
   dropout_2 : 0.4
   nuerons_3 : 80
   activation_3 : relu
   dropout_3 : 0.4
   tuner/epochs : 2
   tuner/initial_epoch : 0
   tuner/bracket : 3
   tuner/round : 0


Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.6313 - loss: 0.6322 - val_accuracy: 0.7446 - val_loss: 0.5654
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6983 - loss: 0.5801 - val_accuracy: 0.7792 - val_loss: 0.5267
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7430 - loss: 0.5426 - val_accuracy: 0.7792 - val_loss: 0.5058
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7449 - loss: 0.5258 - val_accuracy: 0.7662 - val_loss: 0.4926
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/s